# Neteja dels datasets - Filtratge geogràfic per Girona

Aquest notebook filtra tots els datasets que contenen coordenades geogràfiques (lon/lat) per assegurar que només continguin dades de Girona.

## Importacions

In [2]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath("../src"))

from constants import ADD_RAW_OG_DATA, OUTPUT_DATA, PROCESSED_DATA, RAW_OG_DATA
from tool_scripts.geographic_tools import add_cusec_column, crear_mapa

## Filtratge per Girona

Per cada dataset amb coordenades:
1. Afegim la columna `census_tract` utilitzant la funció `add_cusec_column()`
2. Filtrem per mantenir només les files amb `census_tract` que comencen per "17079" (codi de Girona)
3. Mostrem estadístiques abans i després del filtratge

In [13]:
df_cedules_habitabilitat = pd.read_csv(
    f"{ADD_RAW_OG_DATA}/girona_cedules_habitabilitat.csv", low_memory=False
).rename(columns={"Latitud": "lat", "Longitud": "lon"})

df_cedules_habitabilitat["lat"] = (
    df_cedules_habitabilitat["lat"].str.replace(",", ".", regex=False).astype(float)
)
df_cedules_habitabilitat["lon"] = (
    df_cedules_habitabilitat["lon"].str.replace(",", ".", regex=False).astype(float)
)

datasets_with_coords = {
    "for_rent_synthetic": pd.read_csv(f"{RAW_OG_DATA}/girona_for_rent_synthetic.csv"),
    "for_rent": pd.read_csv(f"{RAW_OG_DATA}/girona_for_rent.csv"),
    "services": pd.read_csv(f"{RAW_OG_DATA}/girona_services.csv"),
    "cedules_habitabilitat": df_cedules_habitabilitat,
}

# Diccionari per guardar els datasets filtrats
datasets_filtrats = {}

for name, df in datasets_with_coords.items():
    print(f"\n{'='*60}")
    print(f"Processant: {name}")
    print(f"{'='*60}")

    # Nombre de files original
    original_count = len(df)
    print(f"Files originals: {original_count}")

    # Afegir columna census_tract
    df_with_census = add_cusec_column(df, "lat", "lon")

    # Comprovar si hi ha valors nuls a census_tract
    null_census = df_with_census["census_tract"].isna().sum()
    if null_census > 0:
        print(f"⚠️  Files amb census_tract nul: {null_census}")

    # Mostrar distribució de codis de municipi (primers 5 caràcters del census_tract)
    if "census_tract" in df_with_census.columns:
        df_with_census["municipality_code"] = (
            df_with_census["census_tract"].astype(str).str[:5]
        )
        print("\nDistribució per codi de municipi:")
        girona_count = (df_with_census["municipality_code"] == "17079").sum()
        non_girona_count = (df_with_census["municipality_code"] != "17079").sum()
        print(f"  17079 (Girona): {girona_count}")
        print(f"  Altres municipis: {non_girona_count}")

    # Filtrar per Girona (17079)
    df_filtered = df_with_census[
        df_with_census["census_tract"].astype(str).str.startswith("17079")
    ].copy()

    # Eliminar columna temporal municipality_code
    if "municipality_code" in df_filtered.columns:
        df_filtered = df_filtered.drop(columns=["municipality_code"])

    filtered_count = len(df_filtered)
    removed_count = original_count - filtered_count

    print(f"\nFiles després del filtratge: {filtered_count}")
    print(f"Files eliminades (fora de Girona): {removed_count}")
    print(f"Percentatge mantingut: {(filtered_count/original_count)*100:.2f}%")

    # Guardar dataset filtrat
    datasets_filtrats[name] = df_filtered

print(f"\n{'='*60}")
print("RESUM FINAL")
print(f"{'='*60}")
print(f"Datasets processats i filtrats: {len(datasets_filtrats)}")


Processant: for_rent_synthetic
Files originals: 2500

Distribució per codi de municipi:
  17079 (Girona): 2002
  Altres municipis: 498

Files després del filtratge: 2002
Files eliminades (fora de Girona): 498
Percentatge mantingut: 80.08%

Processant: for_rent
Files originals: 136

Distribució per codi de municipi:
  17079 (Girona): 110
  Altres municipis: 26

Files després del filtratge: 110
Files eliminades (fora de Girona): 26
Percentatge mantingut: 80.88%

Processant: services
Files originals: 714

Distribució per codi de municipi:
  17079 (Girona): 712
  Altres municipis: 2

Files després del filtratge: 712
Files eliminades (fora de Girona): 2
Percentatge mantingut: 99.72%

Processant: cedules_habitabilitat
Files originals: 51259
⚠️  Files amb census_tract nul: 995

Distribució per codi de municipi:
  17079 (Girona): 50258
  Altres municipis: 1001

Files després del filtratge: 50258
Files eliminades (fora de Girona): 1001
Percentatge mantingut: 98.05%

RESUM FINAL
Datasets proces

## Visualització de les localitzacions filtrades

Creem mapes per visualitzar les localitzacions després del filtratge.

In [4]:
output_dir = f"{OUTPUT_DATA}/filtrats"

# Crear mapes per cada dataset filtrat
for name, df in datasets_filtrats.items():
    if len(df) > 0:
        print(f"Creant mapa per {name}...")
        crear_mapa(df, output_dir=output_dir, name_html=f"{name}_filtrat_girona.html")
        print(f"✓ Mapa guardat: {output_dir}/{name}_filtrat_girona.html")
    else:
        print(f"⚠️  {name} no té dades per crear mapa")

Creant mapa per for_rent_synthetic...
✓ Mapa guardat: ../outputs/filtrats/for_rent_synthetic_filtrat_girona.html
Creant mapa per for_rent...
✓ Mapa guardat: ../outputs/filtrats/for_rent_filtrat_girona.html
Creant mapa per services...
✓ Mapa guardat: ../outputs/filtrats/services_filtrat_girona.html


## Guardar datasets filtrats amb els noms originals

In [ ]:
if "for_rent_synthetic" in datasets_filtrats:
    datasets_filtrats["for_rent_synthetic"].to_csv(
        f"{PROCESSED_DATA}/girona_for_rent_synthetic.csv", index=False
    )
    print(
        f"✓ girona_for_rent_synthetic.csv guardat: {len(datasets_filtrats['for_rent_synthetic'])} files"
    )

if "for_rent" in datasets_filtrats:
    datasets_filtrats["for_rent"].to_csv(
        f"{PROCESSED_DATA}/girona_for_rent.csv", index=False
    )
    print(f"✓ girona_for_rent.csv guardat: {len(datasets_filtrats['for_rent'])} files")

if "services" in datasets_filtrats:
    datasets_filtrats["services"].to_csv(
        f"{PROCESSED_DATA}/girona_services.csv", index=False
    )
    print(f"✓ girona_services.csv guardat: {len(datasets_filtrats['services'])} files")

# També guardar datasets que no necessitaven filtratge
df_sociodemographic = pd.read_csv(f"{RAW_OG_DATA}/girona_sociodemographic.csv")
df_energy_certificates = pd.read_csv(f"{RAW_OG_DATA}/girona_energy_certificates.csv")

df_energy_certificates.to_csv(
    f"{PROCESSED_DATA}/girona_energy_certificates.csv", index=False
)
print(f"✓ girona_energy_certificates.csv guardat: {len(df_energy_certificates)} files")

df_sociodemographic.to_csv(f"{PROCESSED_DATA}/girona_sociodemographic.csv", index=False)
print(f"✓ girona_sociodemographic.csv guardat: {len(df_sociodemographic)} files")

print(f"\n✓ Tots els datasets guardats a {PROCESSED_DATA}/")

✓ girona_for_rent_synthetic.csv guardat: 2002 files
✓ girona_for_rent.csv guardat: 110 files
✓ girona_services.csv guardat: 712 files
✓ girona_energy_certificates.csv guardat: 18235 files
✓ girona_sociodemographic.csv guardat: 552 files

✓ Tots els datasets guardats a ../data/processed/
